# 🖼️ JoyCaption Image API — Gradio (Colab)

Simple Gradio web app + API that accepts a single image and returns a concise caption using LLaVA JoyCaption.

- UI: Upload an image → get caption
- API (Blocks):
  - Streaming: POST /gradio_api/caption
  - Enqueue: POST /gradio_api/call/caption → {event_id}, then GET /gradio_api/result/{event_id}
- Base64 helper endpoint: POST /gradio_api/caption_b64 with a data URL string

Note: Use the printed Gradio share URL for external calls, or the Colab proxy URL for session-only access.


In [ ]:
# GPU check
!nvidia-smi || echo 'No NVIDIA GPU visible'

import sys, platform
print(f'Python: {sys.version.split()[0]} | Platform: {platform.platform()}')


In [ ]:
# Install dependencies
!pip -q install 'transformers>=4.44.0' accelerate pillow gradio > /dev/null
import gradio as gr
print('✅ Deps installed')


In [ ]:
# Environment configuration
import os, torch
os.environ['TRANSFORMERS_NO_TORCHVISION'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,max_split_size_mb:128'
if torch.cuda.is_available():
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(False)
    torch.backends.cuda.enable_math_sdp(True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ Torch {torch.__version__} | CUDA: {torch.cuda.is_available()} | Device: {DEVICE}')


In [ ]:
# Load JoyCaption model
from transformers import AutoProcessor, LlavaForConditionalGeneration
from PIL import Image

MODEL_NAME = 'fancyfeast/llama-joycaption-alpha-two-hf-llava'
print(f'📥 Loading model: {MODEL_NAME}')
processor = AutoProcessor.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=(torch.float16 if DEVICE=='cuda' else torch.float32),
    device_map='auto',
    attn_implementation='eager',
    trust_remote_code=True,
)
model.eval()
print(f'✅ Model ready on {DEVICE}')


In [ ]:
# Helpers and captioning
import io, base64, re, time, logging, sys
from typing import Optional

logging.basicConfig(level=logging.INFO, stream=sys.stdout, format='%(asctime)s %(levelname)s %(message)s')
logger = logging.getLogger('joycaption')

MAX_SIDE_DEFAULT = 672

def downscale_image(img: Image.Image, max_side: int = MAX_SIDE_DEFAULT) -> Image.Image:
    w, h = img.size
    if max(w, h) <= max_side:
        return img
    scale = max_side / float(max(w, h))
    return img.resize((int(w*scale), int(h*scale)), Image.BICUBIC)

def _normalize_inputs_for_generate(inputs, device):
    fixed = {}
    for k, v in inputs.items():
        if not hasattr(v, 'to'):
            fixed[k] = v
            continue
        if k == 'pixel_values':
            want = (torch.float16 if device=='cuda' else torch.float32)
            if v.dtype != want:
                v = v.to(want)
        elif k == 'input_ids':
            if v.dtype != torch.int64:
                v = v.to(torch.int64)
        elif k in ('attention_mask','pixel_attention_mask','cross_attention_mask'):
            if v.dtype != torch.int64:
                v = v.to(torch.int64)
        v = v.to(device)
        fixed[k] = v
    return fixed

def caption_single_image(img: Image.Image, prompt: Optional[str] = None, *, max_new_tokens=96, temperature=0.6, top_p=0.9, max_side=MAX_SIDE_DEFAULT) -> str:
    if prompt is None:
        prompt = 'Write a concise, descriptive caption for this image.'
    img = img.convert('RGB')
    img = downscale_image(img, max_side=max_side)

    convo = [
        {'role': 'system', 'content': 'You are a concise, visual captioner.'},
        {'role': 'user',   'content': prompt},
    ]
    tmpl = processor.apply_chat_template(convo, tokenize=False, add_generation_prompt=True)
    raw_inputs = processor(text=[tmpl], images=[img], return_tensors='pt', padding=True)
    inputs = _normalize_inputs_for_generate(raw_inputs, DEVICE)

    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=True, temperature=temperature, top_p=top_p, use_cache=True
        )[0]
    out = out[inputs['input_ids'].shape[1]:]
    text = processor.tokenizer.decode(out, skip_special_tokens=True, clean_up_tokenization_spaces=False).strip()
    return text

DATA_URL_RE = re.compile(r'^data:.*?;base64,(.*)$')
def decode_base64_image(data: str) -> Image.Image:
    m = DATA_URL_RE.match(data or '')
    if m:
        data = m.group(1)
    b = base64.b64decode(data)
    return Image.open(io.BytesIO(b)).convert('RGB')

def caption_image_ui(img: Image.Image) -> dict:
    t0 = time.time()
    logger.info('UI request received')
    cap = caption_single_image(img)
    if DEVICE=='cuda':
        torch.cuda.empty_cache()
    return {'caption': cap, 'elapsed_sec': round(time.time()-t0,3)}

def caption_image_b64_api(b64: str) -> dict:
    t0 = time.time()
    logger.info('API b64 request received')
    img = decode_base64_image(b64)
    cap = caption_single_image(img)
    if DEVICE=='cuda':
        torch.cuda.empty_cache()
    return {'caption': cap, 'elapsed_sec': round(time.time()-t0,3)}


In [ ]:
# Gradio app (UI + API)
with gr.Blocks() as demo:
    gr.Markdown('# JoyCaption — Image Caption Demo')
    with gr.Row():
        img = gr.Image(type='pil', label='Upload an image')
        out = gr.JSON(label='Result')
    btn = gr.Button('Caption')
    btn.click(fn=caption_image_ui, inputs=img, outputs=out, api_name='caption')

    gr.Markdown('Base64 helper (for raw REST without upload tokens)')
    b64 = gr.Textbox(label='data URL (data:image/jpeg;base64,...)')
    out2 = gr.JSON(label='Result (b64)')
    btn2 = gr.Button('Caption (base64)')
    btn2.click(fn=caption_image_b64_api, inputs=b64, outputs=out2, api_name='caption_b64')

demo.launch(server_name='0.0.0.0', server_port=8000, share=True, quiet=False)
print('✅ Launched — see logs above for requests')


In [ ]:
# Colab proxy URL (session-only)
try:
    from google.colab import output as colab_output
    proxy_base = colab_output.eval_js('google.colab.kernel.proxyPort(8000)')
    if proxy_base and isinstance(proxy_base, str):
        if not proxy_base.endswith('/'):
            proxy_base += '/'
        print('🔗 Colab proxy:', proxy_base)
        print('   UI:', proxy_base)
        print('   API (stream):', proxy_base + 'gradio_api/caption')
        print('   API (call):  ', proxy_base + 'gradio_api/call/caption')
        print('   API (b64):   ', proxy_base + 'gradio_api/caption_b64')
    else:
        print('Colab proxy unavailable')
except Exception as e:
    print('Not in Colab or proxy error:', e)


### Usage notes
- Public share URL (printed by launch) is recommended for external calls.
- For REST without file-token upload, use the base64 endpoint: POST /gradio_api/caption_b64 with a data URL string in `data`.
- For Blocks streaming: POST /gradio_api/caption (SSE).
- For two-step queue: POST /gradio_api/call/caption → {event_id}, then GET /gradio_api/result/{event_id}.
